# __Master Pipeline__

# **0. Librairies**

In [26]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
from sklearn.metrics import confusion_matrix


sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset
from lib.data.preprocessing import TorchPreprocessor
from lib.data.train_val_split import train_val_split
from lib.data.preprocessing import TorchPreprocessor
from lib.data.data_augmentation import data_augmented_loader
from lib.utils.model_saver import ModelSaver

In [ ]:
## CONFIGURATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_EPOCHS = 25
LR = 1e-4

DROPOUT = 0.4
WEIGHT_DECAY = 1e-4

USER_NAME = 'Maignan'

NUM_CLASSES = 50

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

## **1. Preprocessing**

In [28]:
train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224))

Train prêt : 6417 images (Pas d'augmentation)
Val prête  : 1582 images (sans augmentation)


In [ ]:
# --- Récupérer tous les labels du val_loader ---
all_labels = []

for _, labels in val_loader:
    all_labels.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

# --- Compter le nombre d'images par classe ---
class_counts_val = pd.Series(all_labels).value_counts().sort_index()

# --- Créer un DataFrame ---
df_val_counts = pd.DataFrame({
    "class": class_counts_val.index,
    "num_images": class_counts_val.values
})

# --- Sauvegarder dans un CSV ---
csv_val_path = "val_class_counts.csv"
df_val_counts.to_csv(csv_val_path, index=False)
print(f"CSV des images par classe dans la validation sauvegardé dans : {csv_val_path}")

## **2. Modèle**

In [ ]:
val_class_counts = pd.read_csv("val_class_counts.csv")
weights = 1.0 / val_class_counts["num_images"].values

In [ ]:
class ResnetFineTune (nn.Module):
    def __init__(self, num_classes, 
                 dropout, 
                 freeze = False):
        
        super(ResnetFineTune, self).__init__()

        self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        if freeze:
            for param in self.resnet.parameters():
                param.requires_grad = False

            for param in self.resnet.layer4.parameters():
                param.requires_grad = True

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward (self, x):
        return self.resnet(x)
    
    def inference(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            probs = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
        
        return pred_class

In [ ]:
model = ResnetFineTune(NUM_CLASSES, DROPOUT)
model.to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/alexm/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:04<00:00, 22.3MB/s]


ResnetFineTune(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
    

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
ModelSaver = ModelSaver(model=model, username=USER_NAME)
ModelSaver.save_training_config(model, optimizer, BATCH_SIZE, NUM_EPOCHS, LR, DEVICE, criterion=criterion)

## **3. F1-score**

In [ ]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

def compute_all_metrics(all_labels, all_preds, num_classes):
    """
    Calcule toutes les métriques en utilisant ta fonction compute_f1 personnalisée.
    """
    all_preds_np = np.array(all_preds)
    all_labels_np = np.array(all_labels)
    acc = np.mean(all_preds_np == all_labels_np)

    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    precision_per_class = precision_score(all_labels, all_preds, average=None, labels=range(num_classes), zero_division=0)
    recall_per_class = recall_score(all_labels, all_preds, average=None, labels=range(num_classes), zero_division=0)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_per_class": f1_per_class,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class
    }

In [ ]:
def compute_confusion_matrix(model, data_loader, device="cpu", num_classes=None):
    model.to(device)
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    if num_classes is None:
        num_classes = max(max(all_labels), max(all_preds)) + 1

    cm = confusion_matrix(all_labels, all_preds)
    return cm

## **4. Fonctions de training et validation**

In [ ]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    all_metrics = compute_all_metrics(all_labels, all_preds, NUM_CLASSES)
    ModelSaver.save_epoch(epoch, all_metrics, mode="train")

    f1_macro = all_metrics["f1_macro"]
    f1_per_class = all_metrics["f1_per_class"]

    # 🔹 Affichage
    print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [ ]:
import torch

def validate(epoch):
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    all_metrics = compute_all_metrics(all_labels, all_preds, NUM_CLASSES)
    ModelSaver.save_epoch(epoch, all_metrics, mode="val")

    f1_macro = all_metrics["f1_macro"]
    f1_per_class = all_metrics["f1_per_class"]

    return (total_loss / len(val_loader), f1_macro, f1_per_class, np.array(all_preds), np.array(all_labels))

## **5. Entrainement**

**Entrainement**

In [ ]:
import csv
import pandas as pd
from sklearn.metrics import confusion_matrix

best_f1 = 0.0

csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro',
              'val_loss', 'val_f1_macro']


loss_train, loss_val = [], []

for epoch in range(NUM_EPOCHS):

    # ===== TRAIN =====
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch(epoch)
    loss_train.append(train_loss)
    # scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")

    # ===== VALIDATION =====
    val_loss, val_f1_macro, val_f1_per_class, val_preds, val_labels = validate(epoch)
    loss_val.append(val_loss)
    
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val F1 Macro: {val_f1_macro:.4f}")



    # ===== BEST MODEL =====
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro

        ModelSaver.save_model(model, name=f"best_model_epoch_{epoch}.pth")

        cm = compute_confusion_matrix(model, val_loader, device=DEVICE, num_classes=NUM_CLASSES)
        ModelSaver.save_confusion_matrix(cm, name=f"confusion_matrix_epoch_{epoch}.csv")        

        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")


       

100%|██████████| 201/201 [00:40<00:00,  4.91it/s]


Train F1 macro: 0.1429

Epoch 1/25
Train Loss: 2.1048 | Train Acc: 0.4458
Train F1 Macro: 0.1429
Val Loss: 1.2882
Val F1 Macro: 0.2252
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.2252


 63%|██████▎   | 126/201 [00:26<00:15,  4.70it/s]


KeyboardInterrupt: 

In [ ]:
plt.plot(loss_train, 'r' , marker='o', linestyle='-', label = 'loss_train')
plt.plot(loss_val, 'b' , marker='o', linestyle='-', label = 'loss_test')
plt.xlabel('Epoch')
plt.ylabel('Loss fonction')
plt.grid()
plt.legend()
plt.show()